# sympy-based Jacobian function builder

2024.10.01

In [1]:
import sympy as sym
from sympy.printing import ccode, print_ccode

In [2]:
def _sym_norm_length3(sym_vec):
    """Symbolic norm alias for length-3 vector"""
    return sym.sqrt(sym_vec[0]**2 + sym_vec[1]**2 + sym_vec[2]**2)


def _sym_dot_length3(sym_vec0, sym_vec1):
    """Symbolic dot product alias for length-3 vector"""
    return sym_vec0[0]*sym_vec1[0] + sym_vec0[1]*sym_vec1[1] + sym_vec0[2]*sym_vec1[2]


def _sym_cross_prod(sym_vec0, sym_vec1):
    return sym.Matrix([
        sym_vec0[1]*sym_vec1[2] - sym_vec0[2]*sym_vec1[1],
        sym_vec0[2]*sym_vec1[0] - sym_vec0[0]*sym_vec1[2],
        sym_vec0[0]*sym_vec1[1] - sym_vec0[1]*sym_vec1[0],
    ])

def _symbolicbattin_3rd_body_perturbation(mu, rvec, svec):
    """Symbolic version of Battin's formula for computing third-body perturbation acceleration"""
    # relative vector
    dvec = rvec - svec
    # q (scalar)
    q = _sym_dot_length3(rvec, rvec - 2*svec)/_sym_dot_length3(svec, svec)
    # Battin's F(q) function
    F = q*((3 + 3*q + q**2)/(1 + sym.sqrt(1 + q)**3))
    return -mu/_sym_norm_length3(dvec)**3 * (rvec + F*svec)


def _symbolicj2_equator_frame_to_inertial_frame(rvec, mu_r3, k_j2, T_IE):
    """Compute acceleration due to J2 in equator-based frame and convert to inertial frame"""
    # convert position vector from inertial to equator frame
    rvec_spacecraft_equator = T_IE.T * rvec  #np.dot(T_IE.T, rvec)
    r = _sym_norm_length3(rvec_spacecraft_equator)
    # compute constants
    k_j2_re2 = k_j2 / r**2
    z2_r2 = (rvec_spacecraft_equator[2] / r)**2
    # copmute acceleration (in equator frame)
    a_j2_equator = -mu_r3 * sym.Matrix([
        rvec_spacecraft_equator[0]*k_j2_re2 * (1 - 5*z2_r2),
        rvec_spacecraft_equator[1]*k_j2_re2 * (1 - 5*z2_r2),
        rvec_spacecraft_equator[2]*k_j2_re2 * (3 - 5*z2_r2),
    ])
    # convert acceleration from equator to inertial frame
    return T_IE * a_j2_equator   #np.dot(T_IE, a_j2_equator)


def _symbolic_j2_using_earth_plane(x, mu, k_J2, earth): 
    """Compute J2 in the form given in Caltech lecture notes"""   
    perp_z_vec      = _sym_cross_prod(earth[0:3], earth[3:6])
    perp_vec        = -_sym_cross_prod(earth[0:3], perp_z_vec)
    M_sc_vec        = x[0:3]
    M_sc            = _sym_norm_length3(M_sc_vec)
    M_sc_perp       = (perp_vec/_sym_norm_length3(perp_vec)) * _sym_dot_length3(M_sc_vec, perp_vec)/_sym_norm_length3(perp_vec)
    M_sc_plane      = sym.Matrix([
        M_sc_vec[0] - M_sc_perp[0], M_sc_vec[1] - M_sc_perp[1], M_sc_vec[2] - M_sc_perp[2]
    ])
    lmbda           = sym.acos(_sym_dot_length3(M_sc_plane, earth[0:3])/_sym_norm_length3(earth[0:3])/_sym_norm_length3(M_sc_plane)) + 6.68*sym.pi/180
    return mu/(_sym_norm_length3(x[0:3])**3)*((k_J2/M_sc**2)*(3*(sym.sin(lmbda))**2 - 1)) * x[0:3]

In [12]:
# number of bodies
n_mus = 3

# Define states
states = sym.Array(sym.symbols('state:%d' % 6))
x,y,z,vx,vy,vz = states  # unpack
r = sym.sqrt(x**2 + y**2 + z**2)

# Define position vectors of 3rd bodies
pos_3bd_list = [sym.Array([sym.symbols('state_%d_%d' % (i, j)) for j in range(3)]) for i in range(n_mus-1)]

# Define constants for SRP
r_sun = sym.Array(sym.symbols('r_sun_:%d' % 3))  # technically redundant but for redefined easier handling
k_srp = sym.symbols("k_srp")

# Define constants for J2
T_IE = sym.Matrix(sym.symbols('T_IE_0_0:3 T_IE_1_0:3 T_IE_2_0:3')).reshape(3, 3)
k_j2  = sym.symbols("k_j2")

# Define mu's
mus = sym.symbols('mu:%d' % n_mus)

In [13]:
eom_name = "Nbody"

# Define eoms
mu_r3 = mus[0]/r**3
ax = -mu_r3 * x
ay = -mu_r3 * y
az = -mu_r3 * z

# Define third-body perturbation's
for idx in range(n_mus-1):
    avec_3bd = _symbolicbattin_3rd_body_perturbation(
        mus[idx+1], states[0:3], pos_3bd_list[idx],
    )
    ax += avec_3bd[0]
    ay += avec_3bd[1]
    az += avec_3bd[2]

if "SRP" in eom_name:
    # Define SRP
    ax += k_srp * (x - r_sun[0])
    ay += k_srp * (y - r_sun[1])
    az += k_srp * (z - r_sun[2])

if "J2" in eom_name:
    # Define J2
    a_j2 = _symbolicj2_equator_frame_to_inertial_frame(
        sym.Matrix(states[0:3]), mu_r3, k_j2, T_IE
    )
    ax += a_j2[0]
    ay += a_j2[1]
    az += a_j2[2]

# Compute sensitivities
Uxx = sym.Matrix([
    [sym.diff(ax, x), sym.diff(ax, y), sym.diff(ax, z)],
    [sym.diff(ay, x), sym.diff(ay, y), sym.diff(ay, z)],
    [sym.diff(az, x), sym.diff(az, y), sym.diff(az, z)],
])

# # concatenate to form system jacobian
# zero_3by3 = sym.zeros(3,3)
# identity_3by3 = sym.eye(3)
# sensitivity_pos = zero_3by3.col_join(Uxx)
# sensitivity_vel = identity_3by3.col_join(sym.zeros(3,3))
# jac = sensitivity_pos.row_join(sensitivity_vel)

# get gravity gradient terms
Uxx_0_0, Uxx_0_1, Uxx_0_2 = ccode(sym.diff(ax, x)), ccode(sym.diff(ax, y)), ccode(sym.diff(ax, z))
Uxx_1_0, Uxx_1_1, Uxx_1_2 = ccode(sym.diff(ay, x)), ccode(sym.diff(ay, y)), ccode(sym.diff(ay, z))
Uxx_2_0, Uxx_2_1, Uxx_2_2 = ccode(sym.diff(az, x)), ccode(sym.diff(az, y)), ccode(sym.diff(az, z))

In [14]:
string_list_raw = [
    Uxx_0_0, Uxx_0_1, Uxx_0_2,
    Uxx_1_0, Uxx_1_1, Uxx_1_2,
    Uxx_2_0, Uxx_2_1, Uxx_2_2,
]
string_list = []
for idx, string in enumerate(string_list_raw):
    # replace body by [i]
    for i in range(n_mus):
        string = string.replace(f"mu{i}", f"mus[{i}]")
        string = string.replace(f"state_{i}_0", f"state_b{i}[0]")
        string = string.replace(f"state_{i}_1", f"state_b{i}[1]")
        string = string.replace(f"state_{i}_2", f"state_b{i}[2]")

    # replace state by compnent
    string = string.replace("state0", "state[0]")
    string = string.replace("state1", "state[1]")
    string = string.replace("state2", "state[2]")

    # transformation matrix notation switch
    string = string.replace("T_IE_0_0", "T_IE[0][0]")
    string = string.replace("T_IE_0_1", "T_IE[0][1]")
    string = string.replace("T_IE_0_2", "T_IE[0][2]")
    string = string.replace("T_IE_1_0", "T_IE[1][0]")
    string = string.replace("T_IE_1_1", "T_IE[1][1]")
    string = string.replace("T_IE_1_2", "T_IE[1][2]")
    string = string.replace("T_IE_2_0", "T_IE[2][0]")
    string = string.replace("T_IE_2_1", "T_IE[2][1]")
    string = string.replace("T_IE_2_2", "T_IE[2][2]")
    
    string = string.replace("r_sun_0", "r_sun[0]")
    string = string.replace("r_sun_1", "r_sun[1]")
    string = string.replace("r_sun_2", "r_sun[2]")
    string_list.append(string)

In [19]:
# write to file
header = """/*
Auto-generated Jacobian function with sympy 
*/

#include <math.h>

"""

if eom_name == "Nbody":
    with open(f"./gravity_gradient_Nbody.cc", "w+") as file:
        file.write(header)
        file.write(f"""void gravity_gradient_N{n_mus}body(
            const double state[42],
            double mus[{n_mus}],
            double T_IE[3][3],
            double state_b0[3],
            double state_b1[3],
            double Uxx[3][3] ) {\n\n""")
        file.write("	// Automatically generated expressions for Uxx\n")
        file.write(f"	Uxx[0][0] = {string_list[0]};\n")
        file.write(f"	Uxx[0][1] = {string_list[1]};\n")
        file.write(f"	Uxx[0][2] = {string_list[2]};\n\n")
        
        file.write(f"	Uxx[1][0] = {string_list[3]};\n")
        file.write(f"	Uxx[1][1] = {string_list[4]};\n")
        file.write(f"	Uxx[1][2] = {string_list[5]};\n\n")
        
        file.write(f"	Uxx[2][0] = {string_list[6]};\n")
        file.write(f"	Uxx[2][1] = {string_list[7]};\n")
        file.write(f"	Uxx[2][2] = {string_list[8]};\n")
        file.write("};")
        
elif eom_name == "NbodySRPJ2":
    with open(f"./gravity_gradient_N{n_mus}bodySRPJ2.cc", "w+") as file:
        file.write(header)
        file.write("""void gravity_gradient_Nbody(
            const double state[42],
            double mus[3],
            double T_IE[3][3],
            double state_b0[3],
            double state_b1[3],
            double k_j2,
            double k_srp,
            double r_sun[3],
            double Uxx[3][3] ) {\n\n""")
        file.write("	// Automatically generated expressions for Uxx\n")
        file.write(f"	Uxx[0][0] = {string_list[0]};\n")
        file.write(f"	Uxx[0][1] = {string_list[1]};\n")
        file.write(f"	Uxx[0][2] = {string_list[2]};\n\n")
        
        file.write(f"	Uxx[1][0] = {string_list[3]};\n")
        file.write(f"	Uxx[1][1] = {string_list[4]};\n")
        file.write(f"	Uxx[1][2] = {string_list[5]};\n\n")
        
        file.write(f"	Uxx[2][0] = {string_list[6]};\n")
        file.write(f"	Uxx[2][1] = {string_list[7]};\n")
        file.write(f"	Uxx[2][2] = {string_list[8]};\n")
        file.write("};")